# STIR-Net V1 — Notebook 30
## Evening staged overfit + trained oracle decomposition

This notebook is the follow-up to Notebook 29.

It is designed for the current STIR-Net V1 repository after the local-mask memory/AMP hardening commit:

`3cafb104d840f1fe9c3801482968967f334496e7`

### Main objective

Give the **current architecture a fair, real training opportunity** before making further architectural conclusions.

The run uses the real temporal graph/history input and warm-starts from the best useful previous spatial/query checkpoint. It then trains:

```text
warm-start
   ↓
query + temporal refresh
   ↓
local-mask bootstrap
   ↓
full joint training
   ↓
trained oracle ladder
```

### Intended schedule

The notebook targets **22:00 Sri Lanka time** and reserves the final part of the run for the oracle decomposition.

For a start around 16:15–16:30:

- query / temporal refresh: ~30 min
- local-mask bootstrap: ~2 h
- joint training: remaining training budget
- trained oracle decomposition: final ~45 min

The exact number of optimizer steps is **wall-clock controlled**, not guessed in advance.

### Why this is different from Notebook 29

Notebook 29 revealed that the local native mask decoder had not actually trained because all matched local Conv3D autograd graphs were retained and the first bootstrap step OOMed.

The current implementation now:

- samples at most `2` matched proposal-local masks per batch item during training;
- streams exhaustive evaluation;
- keeps original proposal-count weighting;
- supports rendering FP16/BF16 cached features outside autocast;
- preserves immutable-anchor proposal center refinement.

This notebook therefore treats the mask and temporal results **after training** as the meaningful evidence.

### Reliability / unattended behavior

The notebook:

- checks the Git SHA and warm-start checkpoint before the long run;
- does an FP16 forward + outside-autocast render preflight;
- logs every optimizer step to JSONL immediately;
- writes a heartbeat file every step;
- saves recovery checkpoints periodically;
- saves stage-boundary and stage-best checkpoints;
- records CUDA allocated/reserved memory and step time;
- catches CUDA OOMs and can automatically fall back from local train cap `2 → 1`;
- isolates post-training oracle gates so one diagnostic failure does not erase the training result;
- does **not** open Napari automatically.

The training result is preserved even if a later oracle diagnostic fails.


In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timedelta, timezone, time as dt_time
from typing import Any, Callable
import copy
import gc
import json
import math
import os
import shutil
import subprocess
import time
import traceback
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from scipy.optimize import linear_sum_assignment

from learned.stirnet import StirNet
from learned.stirnet.debugging.acceptance.first_overfit import (
    _reduced_config,
    _repo_root,
    build_real_batch,
)
from learned.stirnet.model.losses import local_matched_mask_losses
from learned.stirnet.model.matcher import target_ids
from learned.stirnet.model.query_builder import QUERY_SPATIAL_PROPOSAL
from learned.stirnet.training.checkpoint import load_checkpoint, save_checkpoint
from learned.stirnet.training.trainer import (
    Trainer,
    model_forward_from_batch,
    move_batch_to_device,
)

EXPECTED_HEAD = "3cafb104d840f1fe9c3801482968967f334496e7"

SEED = 40266
SOURCE_ID = 9
AMP_DTYPE = torch.float16

# ------------------------------------------------------------------
# Evening schedule
# ------------------------------------------------------------------
SRI_LANKA_TZ = timezone(timedelta(hours=5, minutes=30))
TARGET_END_HOUR = 22
TARGET_END_MINUTE = 0

# Reserve this final window for the trained oracle decomposition.
ORACLE_RESERVE_MINUTES = 45

# The training budget is dynamically split.
QUERY_FRACTION = 0.10
LOCAL_FRACTION = 0.42

# Hard caps prevent an unexpectedly fast stage from running enormous counts.
QUERY_MAX_STEPS = 500
LOCAL_MAX_STEPS = 2000
JOINT_MAX_STEPS = 500

# Training robustness.
RECOVERY_CHECKPOINT_MINUTES = 15
BEST_CHECKPOINT_MINUTES = 10
LOCAL_TRAIN_CAP = 2
ALLOW_OOM_FALLBACK_TO_ONE = True

# Oracle capacity probe.
G1_CAPACITY_MAX_STEPS = 240
G1_CAPACITY_MAX_MINUTES = 12
G1_CAPACITY_LR = 2e-3
G1_EVAL_EVERY = 20

DEFAULT_EXIST_THRESHOLD = 0.50
SOURCE9_SELECTION_NEIGHBORHOOD_DREF = 1.25
MAX_SOURCE9_RENDER_QUERIES = 32

REQUIRE_WARM_START = True
OPEN_NAPARI_AT_END = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    raise RuntimeError("Notebook 30 requires CUDA.")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True

REPO_ROOT = _repo_root(Path.cwd())
DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

RUN_DIR = (
    REPO_ROOT
    / "runs"
    / "stirnet"
    / "evening"
    / f"30_evening_overfit_{datetime.now(SRI_LANKA_TZ).strftime('%Y%m%d_%H%M%S')}"
)
RUN_DIR.mkdir(parents=True, exist_ok=False)

LOG_PATH = RUN_DIR / "run.log"
RESULTS_JSONL = RUN_DIR / "results.jsonl"
ERRORS_JSONL = RUN_DIR / "errors.jsonl"
TRAIN_JSONL = RUN_DIR / "training.jsonl"
HEARTBEAT_PATH = RUN_DIR / "heartbeat.json"

NOTEBOOK_STARTED_MONO = time.monotonic()
NOTEBOOK_STARTED_DT = datetime.now(SRI_LANKA_TZ)

target_end = datetime.combine(
    NOTEBOOK_STARTED_DT.date(),
    dt_time(TARGET_END_HOUR, TARGET_END_MINUTE),
    tzinfo=SRI_LANKA_TZ,
)
if target_end <= NOTEBOOK_STARTED_DT:
    # Safe fallback for an accidental late-night rerun.
    target_end = NOTEBOOK_STARTED_DT + timedelta(hours=6)

training_end = target_end - timedelta(minutes=ORACLE_RESERVE_MINUTES)

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("Run dir    :", RUN_DIR)
print("Device     :", device)
print("GPU        :", torch.cuda.get_device_name(0))
print("Started    :", NOTEBOOK_STARTED_DT.isoformat())
print("Train until:", training_end.isoformat())
print("Target end :", target_end.isoformat())


In [ ]:
def _jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, np.generic):
        return value.item()
    if torch.is_tensor(value):
        if value.numel() == 1:
            return value.detach().cpu().item()
        return value.detach().cpu().tolist()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def log(message: str) -> None:
    line = f"[{datetime.now(SRI_LANKA_TZ).isoformat(timespec='seconds')}] {message}"
    print(line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")
        handle.flush()


def append_jsonl(path: Path, payload: dict) -> None:
    with path.open("a", encoding="utf-8") as handle:
        json.dump(
            {key: _jsonable(value) for key, value in payload.items()},
            handle,
        )
        handle.write("\n")
        handle.flush()


def record_result(phase: str, **payload) -> None:
    append_jsonl(
        RESULTS_JSONL,
        {
            "time": datetime.now(SRI_LANKA_TZ),
            "phase": phase,
            **payload,
        },
    )


def record_error(phase: str, exc: BaseException) -> None:
    text = "".join(
        traceback.format_exception(type(exc), exc, exc.__traceback__)
    )
    log(f"ERROR [{phase}] {type(exc).__name__}: {exc}")
    append_jsonl(
        ERRORS_JSONL,
        {
            "time": datetime.now(SRI_LANKA_TZ),
            "phase": phase,
            "type": type(exc).__name__,
            "message": str(exc),
            "traceback": text,
        },
    )


def safe_phase(name: str, fn: Callable[[], Any], default=None):
    log(f"START {name}")
    try:
        value = fn()
        log(f"DONE  {name}")
        return value
    except KeyboardInterrupt:
        log(f"INTERRUPTED {name}")
        raise
    except BaseException as exc:
        record_error(name, exc)
        cleanup()
        return default


def cleanup() -> None:
    gc.collect()
    torch.cuda.empty_cache()


def git_head() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=REPO_ROOT,
            text=True,
        ).strip()
    except Exception:
        return "unknown"


def write_heartbeat(**payload) -> None:
    HEARTBEAT_PATH.write_text(
        json.dumps(
            {
                "time": datetime.now(SRI_LANKA_TZ).isoformat(),
                **{key: _jsonable(value) for key, value in payload.items()},
            },
            indent=2,
        ),
        encoding="utf-8",
    )


HEAD = git_head()
log(f"Git HEAD: {HEAD}")
if HEAD != EXPECTED_HEAD:
    log(
        "WARNING: local HEAD differs from the Notebook-30 reference commit "
        f"{EXPECTED_HEAD}. The notebook will continue against local code."
    )

free_gib = shutil.disk_usage(REPO_ROOT.anchor).free / 1024**3
log(f"Free disk: {free_gib:.2f} GiB")
if free_gib < 2.0:
    raise RuntimeError("Less than 2 GiB free disk; refusing unattended overfit.")

record_result(
    "environment",
    git_head=HEAD,
    expected_head=EXPECTED_HEAD,
    gpu=torch.cuda.get_device_name(0),
    free_disk_gib=free_gib,
    start=NOTEBOOK_STARTED_DT,
    training_end=training_end,
    target_end=target_end,
)


## 1. Build the real all-cell overfit scene

In [ ]:
batch_cpu, sample_info = build_real_batch(DATA_DIR)
target = batch_cpu["targets"][0]

gt_labels_native = (
    torch.as_tensor(target["label_map"])
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)
current_labels_native = (
    batch_cpu["instance_labels"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.int32, copy=False)
)
spacing_native = (
    batch_cpu["spacing_um"][0]
    .detach()
    .cpu()
    .numpy()
    .astype(np.float64)
)
dref_um = float(batch_cpu["dref_um"][0])

all_gt_ids = target_ids(target).detach().cpu().numpy().astype(int)
all_gt_centers = (
    torch.as_tensor(target["centers_cellscale"])
    .detach()
    .cpu()
    .float()
)

source9_gt_ids = np.unique(
    gt_labels_native[current_labels_native == SOURCE_ID]
)
source9_gt_ids = source9_gt_ids[source9_gt_ids > 0].astype(int)
source9_id_set = set(source9_gt_ids.tolist())

source9_gt_indices = torch.tensor(
    [
        row
        for row, gt_id in enumerate(all_gt_ids.tolist())
        if int(gt_id) in source9_id_set
    ],
    dtype=torch.long,
)
source9_gt_centers = all_gt_centers[source9_gt_indices]

missing_gt_ids = []
for gt_id in all_gt_ids.tolist():
    overlap = current_labels_native[gt_labels_native == int(gt_id)]
    if not np.any(overlap > 0):
        missing_gt_ids.append(int(gt_id))

noisy_source_ids = []
for source_id in np.unique(current_labels_native):
    if int(source_id) <= 0:
        continue
    overlap = gt_labels_native[current_labels_native == int(source_id)]
    if not np.any(overlap > 0):
        noisy_source_ids.append(int(source_id))

log(
    f"Scene: current={sample_info['current_count']} GT={sample_info['target_count']} "
    f"shape={sample_info['roi_shape']} dref={dref_um:.3f} um"
)
log(f"Source-{SOURCE_ID} GT IDs: {source9_gt_ids.tolist()}")
log(f"Missing GT IDs: {missing_gt_ids}")
log(f"Noisy current sources: {noisy_source_ids}")

if len(source9_gt_ids) != 9:
    log(
        f"WARNING: expected 9 GT cells inside source {SOURCE_ID}; "
        f"found {len(source9_gt_ids)}."
    )

record_result(
    "scene",
    sample_info=sample_info,
    source9_gt_ids=source9_gt_ids.tolist(),
    missing_gt_ids=missing_gt_ids,
    noisy_source_ids=noisy_source_ids,
)


## 2. Helper utilities used by both training diagnostics and the oracle ladder


In [ ]:
NODE_KEYS = {
    "graph_x",
    "tracklet_id",
    "node_instance_grid",
    "node_history_valid",
    "node_observed_ref_um",
    "node_time_offset",
    "node_ids",
    "node_event_features",
}
EDGE_INDEX_KEYS = {
    "graph_edge_index",
    "accepted_association_edge_index",
}
EDGE_ATTR_KEYS = {
    "graph_edge_attr",
    "accepted_association_edge_attr",
}
TRACKLET_KEYS = {
    "temporal_ref_um",
    "temporal_status",
    "temporal_batch",
    "history_support",
    "history_support_valid",
    "history_support_dt",
    "history_support_center_um",
    "history_support_extent_um",
    "best_current_component_id",
    "best_component_overlap",
    "second_best_component_overlap",
}
HYP_EDGE_INDEX_KEYS = {"hypothesis_edge_index"}
HYP_EDGE_ATTR_KEYS = {"hypothesis_edge_attr"}


def make_spatial_only_batch(batch: dict) -> dict:
    out = dict(batch)

    for key in NODE_KEYS | EDGE_ATTR_KEYS | TRACKLET_KEYS | HYP_EDGE_ATTR_KEYS:
        value = out.get(key)
        if torch.is_tensor(value):
            out[key] = value[:0]

    for key in EDGE_INDEX_KEYS | HYP_EDGE_INDEX_KEYS:
        value = out.get(key)
        if torch.is_tensor(value):
            out[key] = value[:, :0]

    return out


def valid_proposal_query_rows(outputs) -> torch.Tensor:
    valid = ~outputs.query_padding_mask[0]
    proposal = outputs.query_types[0] == QUERY_SPATIAL_PROPOSAL
    return torch.nonzero(valid & proposal, as_tuple=False).flatten()


def build_gt_aware_pairing(
    outputs,
    gt_ids_ordered: np.ndarray,
    gt_centers_ordered: torch.Tensor,
) -> dict:
    qrows = valid_proposal_query_rows(outputs)
    refs = (
        outputs.query_initial_references_cellscale[0, qrows]
        .detach()
        .float()
        .cpu()
    )

    if len(qrows) == 0 or len(gt_centers_ordered) == 0:
        return {
            "query_rows": torch.empty(0, dtype=torch.long),
            "gt_rows": torch.empty(0, dtype=torch.long),
            "gt_ids": np.asarray([], dtype=int),
            "distances_dref": torch.empty(0),
        }

    distance = torch.cdist(
        refs,
        gt_centers_ordered.detach().float().cpu(),
    )
    q_np, g_np = linear_sum_assignment(distance.numpy())

    q_local = torch.as_tensor(q_np, dtype=torch.long)
    g_rows = torch.as_tensor(g_np, dtype=torch.long)

    order = torch.argsort(g_rows)
    q_local = q_local[order]
    g_rows = g_rows[order]

    query_rows = qrows.detach().cpu()[q_local]
    matched_distances = distance[q_local, g_rows]

    return {
        "query_rows": query_rows,
        "gt_rows": g_rows,
        "gt_ids": np.asarray(gt_ids_ordered, dtype=int)[g_rows.numpy()],
        "distances_dref": matched_distances,
    }


def proposal_set_metrics(
    refs: torch.Tensor,
    gt_centers: torch.Tensor,
    *,
    prefix: str,
) -> dict:
    refs = refs.detach().float().cpu()
    gt_centers = gt_centers.detach().float().cpu()

    if len(gt_centers) == 0:
        return {
            f"{prefix}_proposal_count": int(len(refs)),
            f"{prefix}_gt_count": 0,
        }

    if len(refs) == 0:
        return {
            f"{prefix}_proposal_count": 0,
            f"{prefix}_gt_count": int(len(gt_centers)),
            f"{prefix}_recall_0p25": 0.0,
            f"{prefix}_recall_0p5": 0.0,
            f"{prefix}_recall_1p0": 0.0,
            f"{prefix}_nearest_mean_dref": float("inf"),
            f"{prefix}_nearest_max_dref": float("inf"),
            f"{prefix}_duplicates_0p5": 0,
        }

    distance = torch.cdist(refs, gt_centers)
    nearest = distance.min(dim=0).values
    counts_0p5 = (distance <= 0.5).sum(dim=0)

    return {
        f"{prefix}_proposal_count": int(len(refs)),
        f"{prefix}_gt_count": int(len(gt_centers)),
        f"{prefix}_recall_0p25": float(
            (nearest <= 0.25).float().mean()
        ),
        f"{prefix}_recall_0p5": float(
            (nearest <= 0.5).float().mean()
        ),
        f"{prefix}_recall_1p0": float(
            (nearest <= 1.0).float().mean()
        ),
        f"{prefix}_nearest_mean_dref": float(nearest.mean()),
        f"{prefix}_nearest_max_dref": float(nearest.max()),
        f"{prefix}_duplicates_0p5": int(
            torch.clamp(counts_0p5 - 1, min=0).sum()
        ),
    }


def selection_metrics_for_source9(outputs, threshold: float) -> dict:
    qrows = valid_proposal_query_rows(outputs)
    refs = (
        outputs.query_initial_references_cellscale[0, qrows]
        .detach()
        .float()
        .cpu()
    )
    scores = (
        outputs.exist_logits[0, qrows]
        .sigmoid()
        .detach()
        .float()
        .cpu()
    )

    if len(refs) == 0:
        return {
            "threshold": float(threshold),
            "cluster_candidates": 0,
            "selected": 0,
            "selected_assigned_within_1dref": 0,
            "unique_gt_covered": 0,
            "missing_gt": len(source9_gt_ids),
            "duplicate_selected": 0,
            "exactly_one_gt": 0,
        }

    distance = torch.cdist(refs, source9_gt_centers.float())
    nearest_distance, nearest_gt = distance.min(dim=1)

    cluster = nearest_distance <= SOURCE9_SELECTION_NEIGHBORHOOD_DREF
    selected = cluster & (scores >= float(threshold))

    selected_gt = nearest_gt[selected]
    selected_distance = nearest_distance[selected]
    valid_assignment = selected_distance <= 1.0
    assigned_gt = selected_gt[valid_assignment]

    counts = torch.bincount(
        assigned_gt,
        minlength=len(source9_gt_ids),
    )
    unique_covered = int((counts > 0).sum())

    return {
        "threshold": float(threshold),
        "cluster_candidates": int(cluster.sum()),
        "selected": int(selected.sum()),
        "selected_assigned_within_1dref": int(valid_assignment.sum()),
        "unique_gt_covered": unique_covered,
        "missing_gt": int(len(source9_gt_ids) - unique_covered),
        "duplicate_selected": int(
            torch.clamp(counts - 1, min=0).sum()
        ),
        "exactly_one_gt": int((counts == 1).sum()),
        "mean_selected_exist": (
            float(scores[selected].mean())
            if bool(selected.any())
            else float("nan")
        ),
    }


def eval_local_masks(
    decoder,
    outputs,
    *,
    gt_ids_ordered,
    anchors_cellscale: torch.Tensor,
    query_embeddings: torch.Tensor,
    tag: str,
) -> tuple[pd.DataFrame, dict]:
    rows = []

    for local_row, (gt_id, anchor) in enumerate(
        zip(gt_ids_ordered, anchors_cellscale)
    ):
        query_embedding = query_embeddings[local_row]

        with torch.no_grad():
            prediction = decoder.decode_one(
                outputs.d0_features,
                outputs.spatial_inputs,
                outputs.dense_outputs,
                query_embedding,
                anchor.to(device),
                outputs.spacing_um[0],
                outputs.dref_um[0],
                batch_index=0,
            )

        if prediction.logits is None:
            raise RuntimeError("Local decoder returned no logits.")

        target_crop_cpu = torch.as_tensor(
            gt_labels_native[prediction.slices] == int(gt_id),
            dtype=torch.bool,
        )
        target_crop = target_crop_cpu.to(device)

        support = prediction.support
        probability = prediction.logits.float().sigmoid()
        hard = probability >= 0.5

        support_probability = probability * support.float()
        support_hard = hard & support

        target_total = float(
            np.count_nonzero(gt_labels_native == int(gt_id))
        )
        target_inside_support = float(
            (target_crop & support).sum().detach().cpu()
        )

        intersection_soft = float(
            (support_probability * target_crop.float())
            .sum()
            .detach()
            .cpu()
        )
        predicted_soft = float(
            support_probability.sum().detach().cpu()
        )
        soft_dice_full = (
            2.0 * intersection_soft + 1e-6
        ) / (
            predicted_soft + target_total + 1e-6
        )

        intersection_hard = float(
            (support_hard & target_crop).sum().detach().cpu()
        )
        predicted_hard = float(
            support_hard.sum().detach().cpu()
        )
        hard_dice_full = (
            2.0 * intersection_hard + 1e-6
        ) / (
            predicted_hard + target_total + 1e-6
        )

        rows.append(
            {
                "tag": tag,
                "gt_id": int(gt_id),
                "support_coverage": (
                    target_inside_support / max(target_total, 1.0)
                ),
                "soft_dice_full": soft_dice_full,
                "hard_dice_full": hard_dice_full,
                "hard_volume_ratio": (
                    predicted_hard / max(target_total, 1.0)
                ),
                "target_voxels": int(target_total),
                "predicted_voxels": int(predicted_hard),
            }
        )

    frame = pd.DataFrame(rows)
    summary = {
        "tag": tag,
        "cell_count": len(frame),
        "soft_dice_mean": (
            float(frame["soft_dice_full"].mean())
            if len(frame) else float("nan")
        ),
        "soft_dice_min": (
            float(frame["soft_dice_full"].min())
            if len(frame) else float("nan")
        ),
        "hard_dice_mean": (
            float(frame["hard_dice_full"].mean())
            if len(frame) else float("nan")
        ),
        "hard_dice_min": (
            float(frame["hard_dice_full"].min())
            if len(frame) else float("nan")
        ),
        "support_coverage_min": (
            float(frame["support_coverage"].min())
            if len(frame) else float("nan")
        ),
        "volume_ratio_mean": (
            float(frame["hard_volume_ratio"].mean())
            if len(frame) else float("nan")
        ),
    }
    return frame, summary


## 3. Warm-start and training configuration

In [ ]:
def discover_warm_start() -> Path | None:
    tiers = [
        # Notebook 29's successful query-refresh boundary is preferred if present.
        list(
            REPO_ROOT.glob(
                "runs/stirnet/experiments/29_oracle_ladder_*/checkpoint_query_refresh.pt"
            )
        ),
        list(
            REPO_ROOT.glob(
                "runs/stirnet/experiments/29_oracle_ladder_*/checkpoint_short_train_final.pt"
            )
        ),
        # Otherwise fall back to the known good spatial/query checkpoint.
        list(
            REPO_ROOT.glob(
                "runs/stirnet/overnight/27_overnight_*/checkpoint_best_spatial_query.pt"
            )
        ),
        list(
            REPO_ROOT.glob(
                "runs/stirnet/**/checkpoint_best_spatial_query.pt"
            )
        ),
    ]

    for tier in tiers:
        candidates = [
            path
            for path in set(tier)
            if path.exists() and RUN_DIR not in path.parents
        ]
        if candidates:
            return max(candidates, key=lambda path: path.stat().st_mtime)
    return None


cfg = _reduced_config()
cfg.proposals.enabled = True
cfg.proposals.query_mode = "spatial_proposals"
cfg.local_masks.enabled = True
cfg.local_masks.train_max_queries_per_batch = LOCAL_TRAIN_CAP

# Start in query_bootstrap. We shorten the stage dynamically after its
# wall-clock phase finishes so global_step remains the real optimizer count.
cfg.curriculum.enabled = True
cfg.curriculum.spatial_dense_steps = 0
cfg.curriculum.temporal_dense_steps = 0
cfg.curriculum.query_bootstrap_steps = 1_000_000
cfg.curriculum.native_bootstrap_steps = 1_000_000
cfg.curriculum.joint_spatial_lr_scale = 0.10
cfg.curriculum.joint_dense_lr_scale = 0.50

model = StirNet(cfg)

WARM_START_CHECKPOINT = discover_warm_start()
warm_start_info = {
    "checkpoint": None,
    "loaded": False,
    "migration": [],
}

if WARM_START_CHECKPOINT is None:
    message = (
        "No useful warm-start checkpoint was found. Checked Notebook-29 "
        "query-refresh checkpoints and Notebook-27 best spatial-query checkpoints."
    )
    if REQUIRE_WARM_START:
        raise RuntimeError(message)
    log("WARNING: " + message + " Continuing from fresh initialization.")
else:
    log(f"Warm start candidate: {WARM_START_CHECKPOINT}")
    loaded = load_checkpoint(
        WARM_START_CHECKPOINT,
        model,
        optimizer=None,
        scheduler=None,
        scaler=None,
        map_location="cpu",
        strict=True,
        migrate_history=True,
    )
    warm_start_info = {
        "checkpoint": str(WARM_START_CHECKPOINT),
        "loaded": True,
        "source_step": loaded.get("step"),
        "source_epoch": loaded.get("epoch"),
        "migration": loaded.get("model_migration", []),
    }
    log(
        f"Warm start loaded; source_step={loaded.get('step')} "
        f"migration_notes={len(warm_start_info['migration'])}"
    )

trainer = Trainer(model, cfg, device=device, amp_dtype="fp16")
trainer.global_step = 0

save_checkpoint(
    RUN_DIR / "checkpoint_start.pt",
    model=trainer.model,
    optimizer=trainer.optimizer,
    scheduler=trainer.scheduler,
    scaler=trainer.scaler,
    step=trainer.global_step,
    config=cfg,
    extra={"warm_start": warm_start_info},
)

record_result("warm_start", **warm_start_info)


## 4. Dynamic wall-clock phase schedule

In [ ]:
now = datetime.now(SRI_LANKA_TZ)
training_budget_minutes = max(
    0.0,
    (training_end - now).total_seconds() / 60.0,
)

if training_budget_minutes < 60:
    log(
        "WARNING: less than 60 minutes remain before the planned oracle reserve. "
        "The notebook will still run with proportionally shorter phases."
    )

query_minutes = min(
    30.0,
    max(8.0, training_budget_minutes * QUERY_FRACTION),
)
local_minutes = min(
    120.0,
    max(20.0, training_budget_minutes * LOCAL_FRACTION),
)

# Avoid allocating more than the actual budget.
if query_minutes + local_minutes > training_budget_minutes:
    scale = training_budget_minutes / max(query_minutes + local_minutes, 1e-6)
    query_minutes *= scale
    local_minutes *= scale

query_deadline = now + timedelta(minutes=query_minutes)
local_deadline = min(
    training_end,
    query_deadline + timedelta(minutes=local_minutes),
)
joint_deadline = training_end

schedule = pd.DataFrame(
    [
        {
            "phase": "query_temporal_refresh",
            "planned_start": now,
            "planned_end": query_deadline,
            "planned_minutes": query_minutes,
        },
        {
            "phase": "local_mask_bootstrap",
            "planned_start": query_deadline,
            "planned_end": local_deadline,
            "planned_minutes": max(
                0.0,
                (local_deadline - query_deadline).total_seconds() / 60.0,
            ),
        },
        {
            "phase": "joint",
            "planned_start": local_deadline,
            "planned_end": joint_deadline,
            "planned_minutes": max(
                0.0,
                (joint_deadline - local_deadline).total_seconds() / 60.0,
            ),
        },
        {
            "phase": "oracle",
            "planned_start": training_end,
            "planned_end": target_end,
            "planned_minutes": ORACLE_RESERVE_MINUTES,
        },
    ]
)
display(schedule)
schedule.to_csv(RUN_DIR / "planned_schedule.csv", index=False)


## 5. Critical preflight

This is intentionally small.

It verifies the exact unattended sequence that previously failed:

1. full-temporal FP16 forward,
2. autocast context exits,
3. proposal-local `render_masks()` runs outside autocast.

If this fails, the long run is stopped immediately rather than wasting hours.


In [ ]:
def _critical_preflight():
    trainer.model.eval()
    trainer.criterion.eval()
    gpu_batch = move_batch_to_device(batch_cpu, device)

    cleanup()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()

    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = model_forward_from_batch(
            trainer.model,
            gpu_batch,
            return_debug=False,
        )

    proposals = torch.nonzero(
        (outputs.query_types[0] == QUERY_SPATIAL_PROPOSAL)
        & (~outputs.query_padding_mask[0]),
        as_tuple=False,
    ).flatten()

    if proposals.numel() == 0:
        raise RuntimeError("Preflight produced no spatial proposals.")

    selected = proposals[: min(2, proposals.numel())]
    with torch.no_grad():
        rendered = trainer.model.render_masks(outputs, [selected])

    if not bool(torch.isfinite(rendered[0]).all()):
        raise RuntimeError("Preflight local render contains non-finite values.")

    result = {
        "seconds": time.perf_counter() - started,
        "proposal_count": int(proposals.numel()),
        "render_shape": tuple(rendered[0].shape),
        "render_dtype": str(rendered[0].dtype),
        "peak_allocated_gib": (
            torch.cuda.max_memory_allocated() / 1024**3
        ),
        "peak_reserved_gib": (
            torch.cuda.max_memory_reserved() / 1024**3
        ),
    }

    del rendered, selected, proposals, outputs, gpu_batch
    cleanup()

    log(
        "Preflight passed: "
        f"peak_alloc={result['peak_allocated_gib']:.2f} GiB "
        f"peak_reserved={result['peak_reserved_gib']:.2f} GiB"
    )
    return result


preflight = safe_phase("critical_preflight", _critical_preflight)
if preflight is None:
    raise RuntimeError(
        "Critical preflight failed. Long training was not started. "
        "Inspect errors.jsonl."
    )

record_result("critical_preflight", **preflight)


# Part A — Evening staged overfit

The real temporal graph/history is used throughout training.

`query_bootstrap` already trains the temporal and query groups together, so it serves as the **query + temporal refresh** stage. High-resolution local-mask losses are disabled there.

Then:

- `native_bootstrap`: only the local mask decoder trains;
- `joint`: all groups train with the configured low spatial LR and full local-mask pathway.


In [ ]:
training_rows: list[dict] = []
stage_summaries: list[dict] = []
last_recovery_save = time.monotonic()
last_best_save_by_stage: dict[str, float] = {}


def save_training_checkpoint(
    filename: str,
    *,
    stage: str,
    extra: dict | None = None,
) -> None:
    payload_extra = {
        "stage": stage,
        "warm_start": warm_start_info,
        "local_train_cap": int(
            trainer.model.local_mask_decoder.cfg.train_max_queries_per_batch
        ),
    }
    if extra:
        payload_extra.update(extra)

    save_checkpoint(
        RUN_DIR / filename,
        model=trainer.model,
        optimizer=trainer.optimizer,
        scheduler=trainer.scheduler,
        scaler=trainer.scaler,
        step=trainer.global_step,
        config=cfg,
        extra=payload_extra,
    )


def training_metric_score(stage: str, metrics: dict) -> float:
    # Native bootstrap's total loss is exactly the useful local objective under
    # its loss overrides, so a single rule is sufficient and easy to audit.
    return float(metrics.get("loss", float("inf")))


def gradients_finite() -> tuple[bool, int, int]:
    grads = [
        parameter.grad
        for parameter in trainer.model.parameters()
        if parameter.grad is not None
    ]
    finite = all(bool(torch.isfinite(value).all()) for value in grads)
    nonzero = sum(bool(torch.count_nonzero(value)) for value in grads)
    return finite, nonzero, len(grads)


def persist_training_row(row: dict) -> None:
    training_rows.append(row)
    append_jsonl(TRAIN_JSONL, row)

    write_heartbeat(
        status="training",
        stage=row["stage"],
        global_step=row["global_step"],
        loss=row.get("loss"),
        dice_hi=row.get("dice_hi"),
        focal_hi=row.get("focal_hi"),
        local_sampled=row.get("local_sampled"),
        local_original=row.get("local_original"),
        peak_allocated_gib=row.get("peak_allocated_gib"),
        peak_reserved_gib=row.get("peak_reserved_gib"),
    )


def maybe_save_recovery(stage: str) -> None:
    global last_recovery_save
    elapsed_minutes = (
        time.monotonic() - last_recovery_save
    ) / 60.0
    if elapsed_minutes >= RECOVERY_CHECKPOINT_MINUTES:
        save_training_checkpoint(
            "checkpoint_recovery.pt",
            stage=stage,
        )
        last_recovery_save = time.monotonic()
        log(f"Recovery checkpoint updated during {stage}.")


def maybe_save_best(
    stage: str,
    metrics: dict,
    best: dict,
) -> dict:
    score = training_metric_score(stage, metrics)
    now_mono = time.monotonic()

    if score >= best["score"]:
        return best

    # Update the in-memory best immediately, but rate-limit disk writes.
    best["score"] = score
    best["step"] = trainer.global_step

    last_save = last_best_save_by_stage.get(stage, -float("inf"))
    if (
        (now_mono - last_save) / 60.0
        >= BEST_CHECKPOINT_MINUTES
    ):
        save_training_checkpoint(
            f"checkpoint_{stage}_best.pt",
            stage=stage,
            extra={
                "best_score": score,
                "best_step": trainer.global_step,
            },
        )
        last_best_save_by_stage[stage] = now_mono
        log(
            f"Saved {stage} best checkpoint: "
            f"step={trainer.global_step}, score={score:.5f}"
        )
    return best


def reduce_local_cap_to_one(reason: str) -> bool:
    current = int(
        trainer.model.local_mask_decoder.cfg.train_max_queries_per_batch
    )
    if current <= 1:
        return False

    cfg.local_masks.train_max_queries_per_batch = 1
    trainer.model.local_mask_decoder.cfg.train_max_queries_per_batch = 1

    log(
        "OOM FALLBACK: local-mask train cap changed "
        f"{current} -> 1 because {reason}"
    )
    record_result(
        "oom_fallback",
        reason=reason,
        old_cap=current,
        new_cap=1,
        global_step=trainer.global_step,
    )
    return True


def run_training_phase(
    *,
    phase_name: str,
    deadline: datetime,
    max_steps: int,
) -> dict:
    started_dt = datetime.now(SRI_LANKA_TZ)
    started_mono = time.monotonic()
    successful = 0
    failures = 0
    best = {
        "score": float("inf"),
        "step": None,
    }
    last_metrics = None

    log(
        f"TRAIN {phase_name}: stage_at_start="
        f"{trainer.curriculum.apply(trainer.global_step).name}, "
        f"deadline={deadline.isoformat()}, max_steps={max_steps}"
    )

    while successful < max_steps:
        if datetime.now(SRI_LANKA_TZ) >= deadline:
            break
        if datetime.now(SRI_LANKA_TZ) >= training_end:
            break

        torch.cuda.reset_peak_memory_stats()
        step_started = time.perf_counter()

        retry = False
        try:
            metrics = trainer.train_step(batch_cpu)
        except torch.cuda.OutOfMemoryError as exc:
            failures += 1
            record_error(
                f"train_{phase_name}_oom_step_{trainer.global_step}",
                exc,
            )
            trainer.optimizer.zero_grad(set_to_none=True)
            cleanup()

            if (
                ALLOW_OOM_FALLBACK_TO_ONE
                and phase_name in {"local_mask_bootstrap", "joint"}
                and reduce_local_cap_to_one(
                    f"{phase_name} step {trainer.global_step}"
                )
            ):
                retry = True
            else:
                log(
                    f"Stopping {phase_name} after unrecoverable CUDA OOM."
                )
                break
        except KeyboardInterrupt:
            save_training_checkpoint(
                "checkpoint_interrupted.pt",
                stage=phase_name,
            )
            raise
        except BaseException as exc:
            failures += 1
            record_error(
                f"train_{phase_name}_step_{trainer.global_step}",
                exc,
            )
            trainer.optimizer.zero_grad(set_to_none=True)
            cleanup()
            save_training_checkpoint(
                "checkpoint_error_recovery.pt",
                stage=phase_name,
                extra={"error": str(exc)},
            )
            log(
                f"Stopping {phase_name} after a non-OOM training error."
            )
            break

        if retry:
            torch.cuda.reset_peak_memory_stats()
            step_started = time.perf_counter()
            try:
                metrics = trainer.train_step(batch_cpu)
            except BaseException as exc:
                failures += 1
                record_error(
                    f"train_{phase_name}_retry_step_{trainer.global_step}",
                    exc,
                )
                trainer.optimizer.zero_grad(set_to_none=True)
                cleanup()
                break

        torch.cuda.synchronize()
        step_seconds = time.perf_counter() - step_started
        successful += 1
        last_metrics = metrics

        sampled = getattr(
            trainer.criterion,
            "last_local_sampled_requests",
            [],
        )
        original = getattr(
            trainer.criterion,
            "last_local_original_request_counts",
            [],
        )

        finite = True
        grad_nonzero = None
        grad_total = None
        if successful == 1 or successful % 10 == 0:
            finite, grad_nonzero, grad_total = gradients_finite()
            if not finite:
                log(
                    f"WARNING: non-finite gradients detected in {phase_name}."
                )

        row = {
            "time": datetime.now(SRI_LANKA_TZ).isoformat(),
            "requested_phase": phase_name,
            "stage": trainer.curriculum_stage.name,
            "global_step": trainer.global_step,
            "phase_successful_step": successful,
            "step_seconds": step_seconds,
            "peak_allocated_gib": (
                torch.cuda.max_memory_allocated() / 1024**3
            ),
            "peak_reserved_gib": (
                torch.cuda.max_memory_reserved() / 1024**3
            ),
            "local_train_cap": int(
                trainer.model.local_mask_decoder.cfg.train_max_queries_per_batch
            ),
            "local_sampled": len(sampled),
            "local_original": (
                int(sum(original))
                if original else 0
            ),
            "gradients_finite": finite,
            "grad_nonzero": grad_nonzero,
            "grad_total": grad_total,
            **metrics,
        }
        persist_training_row(row)

        best = maybe_save_best(
            phase_name,
            metrics,
            best,
        )
        maybe_save_recovery(phase_name)

        if successful == 1 or successful % 5 == 0:
            log(
                f"{phase_name} n={successful:04d} global={trainer.global_step:05d} "
                f"loss={metrics.get('loss', float('nan')):.4f} "
                f"dice_hi={metrics.get('dice_hi', float('nan')):.4f} "
                f"coarse={metrics.get('dice_coarse', float('nan')):.4f} "
                f"center={metrics.get('center', float('nan')):.4f} "
                f"sampled={len(sampled)} "
                f"alloc={row['peak_allocated_gib']:.2f}GiB "
                f"reserved={row['peak_reserved_gib']:.2f}GiB "
                f"seconds={step_seconds:.1f}"
            )

    save_training_checkpoint(
        f"checkpoint_{phase_name}_last.pt",
        stage=phase_name,
        extra={
            "successful_steps": successful,
            "failures": failures,
            "best_score_observed": best["score"],
            "best_step_observed": best["step"],
        },
    )
    save_training_checkpoint(
        "checkpoint_recovery.pt",
        stage=phase_name,
    )

    result = {
        "phase": phase_name,
        "started": started_dt,
        "finished": datetime.now(SRI_LANKA_TZ),
        "elapsed_minutes": (
            time.monotonic() - started_mono
        ) / 60.0,
        "successful_steps": successful,
        "failures": failures,
        "global_step": trainer.global_step,
        "best_score_observed": best["score"],
        "best_step_observed": best["step"],
        "last_metrics": last_metrics,
        "local_train_cap": int(
            trainer.model.local_mask_decoder.cfg.train_max_queries_per_batch
        ),
    }
    stage_summaries.append(result)
    record_result("training_phase", **result)
    return result


## 6. Query + temporal refresh

In [ ]:
query_result = run_training_phase(
    phase_name="query_temporal_refresh",
    deadline=query_deadline,
    max_steps=QUERY_MAX_STEPS,
)

# Make the completed query step count the actual curriculum boundary.
query_steps = int(query_result["successful_steps"])
cfg.curriculum.query_bootstrap_steps = query_steps

# Force the controller to recompute stage semantics on the next step.
trainer.curriculum.current = None

save_training_checkpoint(
    "checkpoint_after_query.pt",
    stage="after_query",
    extra={"query_steps": query_steps},
)

log(
    f"Query/temporal refresh complete: {query_steps} steps. "
    f"Next curriculum stage at global_step={trainer.global_step}: "
    f"{trainer.curriculum.apply(trainer.global_step).name}"
)


## 7. Local native-mask bootstrap

In [ ]:
local_result = run_training_phase(
    phase_name="local_mask_bootstrap",
    deadline=local_deadline,
    max_steps=LOCAL_MAX_STEPS,
)

local_steps = int(local_result["successful_steps"])
cfg.curriculum.native_bootstrap_steps = local_steps
trainer.curriculum.current = None

save_training_checkpoint(
    "checkpoint_after_local_mask.pt",
    stage="after_local_mask",
    extra={
        "query_steps": query_steps,
        "local_steps": local_steps,
    },
)

next_stage = trainer.curriculum.apply(trainer.global_step).name
log(
    f"Local-mask bootstrap complete: {local_steps} steps. "
    f"Next stage: {next_stage}"
)

if local_steps == 0:
    log(
        "WARNING: zero local bootstrap steps succeeded. "
        "Joint training will still be attempted, but post-training mask "
        "conclusions must be treated cautiously."
    )


## 8. Full joint training until the oracle reserve

In [ ]:
joint_result = run_training_phase(
    phase_name="joint",
    deadline=joint_deadline,
    max_steps=JOINT_MAX_STEPS,
)

save_training_checkpoint(
    "checkpoint_joint_last.pt",
    stage="joint",
    extra={
        "query_steps": query_steps,
        "local_steps": local_steps,
        "joint_steps": int(joint_result["successful_steps"]),
    },
)

save_training_checkpoint(
    "checkpoint_evening_final.pt",
    stage="evening_final",
    extra={
        "stage_summaries": stage_summaries,
        "warm_start": warm_start_info,
    },
)

if training_rows:
    training_df = pd.DataFrame(training_rows)
    training_df.to_csv(
        RUN_DIR / "training.csv",
        index=False,
    )
else:
    training_df = pd.DataFrame()

pd.DataFrame(stage_summaries).to_json(
    RUN_DIR / "stage_summaries.json",
    orient="records",
    indent=2,
    default_handler=str,
)

display(
    pd.DataFrame(
        [
            {
                key: value
                for key, value in summary.items()
                if key != "last_metrics"
            }
            for summary in stage_summaries
        ]
    )
)

log(
    "Training portion complete. "
    f"Current Sri Lanka time: {datetime.now(SRI_LANKA_TZ).isoformat(timespec='seconds')}"
)


## 9. Training curves

In [ ]:
if len(training_df):
    for metric in ("loss", "dice_hi", "focal_hi", "dice_coarse", "center"):
        if metric not in training_df.columns:
            continue

        fig, ax = plt.subplots(figsize=(10, 4))
        for stage_name, frame in training_df.groupby("requested_phase"):
            ax.plot(
                frame["global_step"],
                frame[metric],
                label=stage_name,
            )
        ax.set_xlabel("optimizer step")
        ax.set_ylabel(metric)
        ax.set_title(f"Notebook 30 — {metric}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(
            RUN_DIR / f"training_{metric}.png",
            dpi=140,
        )
        plt.show()
        plt.close(fig)


# Part B — Trained oracle decomposition

The following gates are based on the Notebook-29 oracle ladder, but now use a model that has had a genuine opportunity to train its new local-mask and temporal pathways.

The order is:

```text
GT EVERYTHING
      ↓
G1: can the final local-mask architecture fit masks with GT anchors?
      ↓
G2: actual trained query + GT anchor
      ↓
G3: actual query + predicted immutable proposal anchor
      ↓
G4: proposal pool coverage
      ↓
G5: learned existence/cardinality
      ↓
G6: oracle proposal-score field vs learned score field
      ↓
G7: missing-cell discovery / noisy-current suppression
      ↓
AUX: oracle one-per-GT composition vs learned selection composition
      ↓
G8: full temporal model vs spatial-only ablation
```

Each gate is error-isolated. Training checkpoints are already safe on disk.


In [ ]:
# Stop any remaining training semantics and take one stable full-temporal snapshot.
trainer.model.eval()
trainer.criterion.eval()
cleanup()

def _final_full_forward():
    gpu_batch = move_batch_to_device(batch_cpu, device)
    torch.cuda.reset_peak_memory_stats()

    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        outputs = model_forward_from_batch(
            trainer.model,
            gpu_batch,
            return_debug=True,
            temporal_memory_ablation="full",
            temporal_routing_ablation="full",
        )

    log(
        f"Final full-temporal snapshot: Q={outputs.exist_logits.shape[1]} "
        f"peak={torch.cuda.max_memory_allocated()/1024**3:.2f} GiB"
    )
    del gpu_batch
    return outputs


out_full = safe_phase(
    "oracle_final_full_forward",
    _final_full_forward,
)
if out_full is None:
    raise RuntimeError(
        "Final trained forward failed. Training checkpoints are safe, "
        "but the oracle ladder cannot continue."
    )

source9_pairing = build_gt_aware_pairing(
    out_full,
    source9_gt_ids,
    source9_gt_centers,
)

pairing_df = pd.DataFrame(
    {
        "gt_id": source9_pairing["gt_ids"],
        "query_row": source9_pairing["query_rows"].numpy(),
        "initial_anchor_error_dref": source9_pairing["distances_dref"].numpy(),
    }
)

if len(pairing_df):
    qrows_dev = source9_pairing["query_rows"].to(device)
    grows = source9_pairing["gt_rows"]
    paired_gt = source9_gt_centers[grows]

    pairing_df["source_instance_id"] = (
        out_full.source_instance_ids[0, qrows_dev]
        .detach()
        .cpu()
        .numpy()
    )
    pairing_df["exist_prob"] = (
        out_full.exist_logits[0, qrows_dev]
        .sigmoid()
        .detach()
        .cpu()
        .numpy()
    )
    final_centers = (
        out_full.centers_cellscale[0, qrows_dev]
        .detach()
        .float()
        .cpu()
    )
    pairing_df["final_center_error_dref"] = (
        torch.linalg.vector_norm(
            final_centers - paired_gt,
            dim=-1,
        ).numpy()
    )

display(pairing_df)
pairing_df.to_csv(
    RUN_DIR / "O00_source9_gt_aware_pairing.csv",
    index=False,
)


## G1 — Oracle local-mask capacity with GT anchors + common query

There is no literal ground-truth neural query embedding.

This capacity probe therefore:

- freezes the trained spatial evidence;
- uses the exact GT cell centers as local anchors;
- uses the **same common zero query** for all nine cells;
- optimizes only a temporary copy of the local decoder;
- samples one GT cell per optimizer update;
- uses the real local Dice + focal objective.

The main model is never modified.

If this probe cannot fit the nine masks despite GT localization, that is strong evidence against the local mask/evidence interface.


In [ ]:
gate1_frame = pd.DataFrame()
gate1_summary = None

def _gate1_capacity():
    if len(source9_gt_ids) == 0:
        raise RuntimeError("No source-9 GT cells available.")

    decoder = copy.deepcopy(
        trainer.model.local_mask_decoder
    ).to(device)
    decoder.train()

    optimizer = torch.optim.AdamW(
        decoder.parameters(),
        lr=G1_CAPACITY_LR,
        weight_decay=1e-4,
    )

    common_query = torch.zeros(
        cfg.decoder.d_model,
        device=device,
        dtype=torch.float32,
    )
    gt_anchors = source9_gt_centers.to(device)
    gt_ids = source9_gt_ids.tolist()

    started = time.monotonic()
    hard_deadline = min(
        time.monotonic() + G1_CAPACITY_MAX_MINUTES * 60.0,
        time.monotonic()
        + max(
            60.0,
            (target_end - datetime.now(SRI_LANKA_TZ)).total_seconds() - 5 * 60.0,
        ),
    )

    history = []
    best_score = -1.0
    best_state = None

    for step in range(G1_CAPACITY_MAX_STEPS + 1):
        if time.monotonic() >= hard_deadline:
            break

        if step < G1_CAPACITY_MAX_STEPS:
            decoder.train()
            optimizer.zero_grad(set_to_none=True)

            chosen = int(
                torch.randint(
                    0,
                    len(gt_ids),
                    (1,),
                    device=device,
                ).item()
            )
            gt_id = int(gt_ids[chosen])
            anchor = gt_anchors[chosen]

            prediction = decoder.decode_one(
                out_full.d0_features,
                out_full.spatial_inputs,
                out_full.dense_outputs,
                common_query,
                anchor,
                out_full.spacing_um[0],
                out_full.dref_um[0],
                batch_index=0,
            )
            if prediction.logits is None:
                raise RuntimeError("Oracle decoder returned no logits.")

            target_crop = torch.as_tensor(
                gt_labels_native[prediction.slices] == gt_id,
                dtype=torch.float32,
                device=device,
            )
            support = prediction.support.to(device)
            target_crop = target_crop * support.float()

            dice, focal = local_matched_mask_losses(
                prediction.logits[None],
                target_crop[None],
                support[None],
                alpha=cfg.losses.mask_focal_alpha_pos,
                gamma=cfg.losses.mask_focal_gamma,
            )
            loss = (
                cfg.losses.dice_hi * dice
                + cfg.losses.focal_hi * focal
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(
                decoder.parameters(),
                1.0,
            )
            optimizer.step()
        else:
            loss = torch.tensor(float("nan"))

        if step == 0 or step % G1_EVAL_EVERY == 0:
            decoder.eval()
            query_batch = common_query[None].expand(
                len(gt_ids), -1
            )
            frame, summary = eval_local_masks(
                decoder,
                out_full,
                gt_ids_ordered=gt_ids,
                anchors_cellscale=gt_anchors,
                query_embeddings=query_batch,
                tag="G1_oracle_capacity",
            )
            history.append(
                {
                    "step": step,
                    "elapsed_minutes": (
                        time.monotonic() - started
                    ) / 60.0,
                    "hard_dice_mean": summary["hard_dice_mean"],
                    "hard_dice_min": summary["hard_dice_min"],
                    "soft_dice_mean": summary["soft_dice_mean"],
                    "support_coverage_min": summary[
                        "support_coverage_min"
                    ],
                }
            )
            log(
                f"G1 step={step:04d} "
                f"hardDice={summary['hard_dice_mean']:.4f} "
                f"min={summary['hard_dice_min']:.4f}"
            )

            if summary["hard_dice_mean"] > best_score:
                best_score = summary["hard_dice_mean"]
                best_state = copy.deepcopy(
                    decoder.state_dict()
                )

    if best_state is not None:
        decoder.load_state_dict(best_state)

    decoder.eval()
    query_batch = common_query[None].expand(len(gt_ids), -1)
    frame, summary = eval_local_masks(
        decoder,
        out_full,
        gt_ids_ordered=gt_ids,
        anchors_cellscale=gt_anchors,
        query_embeddings=query_batch,
        tag="G1_oracle_capacity",
    )
    summary["fit_steps_attempted"] = (
        int(history[-1]["step"])
        if history else 0
    )
    summary["best_hard_dice_observed"] = best_score

    pd.DataFrame(history).to_csv(
        RUN_DIR / "O01_G1_capacity_history.csv",
        index=False,
    )
    frame.to_csv(
        RUN_DIR / "O01_G1_capacity_per_cell.csv",
        index=False,
    )

    del decoder
    cleanup()
    return frame, summary


gate1_result = safe_phase(
    "O01_G1_oracle_capacity",
    _gate1_capacity,
)
if gate1_result is not None:
    gate1_frame, gate1_summary = gate1_result
    display(gate1_frame)
    print(gate1_summary)
    record_result(
        "O01_G1_oracle_capacity",
        **gate1_summary,
    )


## G2 — Actual trained query + GT anchor

No diagnostic fitting occurs here.

This tests the **trained query/local-decoder interface under perfect localization**.


In [ ]:
gate2_frame = pd.DataFrame()
gate2_summary = None

def _gate2_actual_query_gt_anchor():
    qrows = source9_pairing["query_rows"]
    grows = source9_pairing["gt_rows"]
    qrows_dev = qrows.to(device)

    gt_ids = source9_pairing["gt_ids"]
    gt_anchors = source9_gt_centers[grows].to(device)
    query_embeddings = (
        out_full.query_embeddings[0, qrows_dev]
        .detach()
    )

    frame, summary = eval_local_masks(
        trainer.model.local_mask_decoder,
        out_full,
        gt_ids_ordered=gt_ids,
        anchors_cellscale=gt_anchors,
        query_embeddings=query_embeddings,
        tag="G2_actual_query_gt_anchor",
    )
    frame.to_csv(
        RUN_DIR / "O02_G2_actual_query_gt_anchor.csv",
        index=False,
    )
    return frame, summary


gate2_result = safe_phase(
    "O02_G2_actual_query_gt_anchor",
    _gate2_actual_query_gt_anchor,
)
if gate2_result is not None:
    gate2_frame, gate2_summary = gate2_result
    display(gate2_frame)
    print(gate2_summary)
    record_result(
        "O02_G2_actual_query_gt_anchor",
        **gate2_summary,
    )


## G3 — Actual query + predicted immutable proposal anchor

GT one-to-one assignment is retained.

This isolates localization/support from query/mask quality and also measures center refinement for the **same paired biological identity**.


In [ ]:
gate3_frame = pd.DataFrame()
gate3_summary = None
center_identity_df = pd.DataFrame()

def _gate3_predicted_anchor():
    qrows = source9_pairing["query_rows"]
    grows = source9_pairing["gt_rows"]
    qrows_dev = qrows.to(device)

    gt_ids = source9_pairing["gt_ids"]
    predicted_anchors = (
        out_full.query_initial_references_cellscale[
            0, qrows_dev
        ].detach()
    )
    query_embeddings = (
        out_full.query_embeddings[
            0, qrows_dev
        ].detach()
    )

    frame, summary = eval_local_masks(
        trainer.model.local_mask_decoder,
        out_full,
        gt_ids_ordered=gt_ids,
        anchors_cellscale=predicted_anchors,
        query_embeddings=query_embeddings,
        tag="G3_predicted_anchor",
    )

    paired_gt = source9_gt_centers[grows]
    final_centers = (
        out_full.centers_cellscale[
            0, qrows_dev
        ].detach().float().cpu()
    )
    initial_centers = (
        predicted_anchors.detach().float().cpu()
    )

    initial_error = torch.linalg.vector_norm(
        initial_centers - paired_gt,
        dim=-1,
    )
    final_error = torch.linalg.vector_norm(
        final_centers - paired_gt,
        dim=-1,
    )
    movement = torch.linalg.vector_norm(
        final_centers - initial_centers,
        dim=-1,
    )

    center_df = pd.DataFrame(
        {
            "gt_id": gt_ids,
            "query_row": qrows.numpy(),
            "initial_error_dref": initial_error.numpy(),
            "final_error_dref": final_error.numpy(),
            "movement_dref": movement.numpy(),
        }
    )

    summary.update(
        {
            "initial_center_error_mean_dref": float(
                initial_error.mean()
            ),
            "final_center_error_mean_dref": float(
                final_error.mean()
            ),
            "center_movement_mean_dref": float(
                movement.mean()
            ),
        }
    )

    frame.to_csv(
        RUN_DIR / "O03_G3_predicted_anchor_masks.csv",
        index=False,
    )
    center_df.to_csv(
        RUN_DIR / "O03_G3_center_identity.csv",
        index=False,
    )
    return frame, summary, center_df


gate3_result = safe_phase(
    "O03_G3_predicted_anchor",
    _gate3_predicted_anchor,
)
if gate3_result is not None:
    gate3_frame, gate3_summary, center_identity_df = gate3_result
    display(gate3_frame)
    display(center_identity_df)
    print(gate3_summary)
    record_result(
        "O03_G3_predicted_anchor",
        **gate3_summary,
    )


## G4 — Proposal-pool coverage

In [ ]:
gate4_summary = None
missing_gt_proposal_df = pd.DataFrame()

def _gate4_proposal_locations():
    proposals = out_full.proposals
    if proposals is None:
        raise RuntimeError("Spatial proposal state is missing.")

    valid = ~proposals.padding_mask[0]
    learned = valid & ~proposals.fallback_mask[0]

    all_refs = (
        proposals.references_cellscale[0, valid]
        .detach()
        .float()
        .cpu()
    )
    learned_refs = (
        proposals.references_cellscale[0, learned]
        .detach()
        .float()
        .cpu()
    )

    summary = {}
    summary.update(
        proposal_set_metrics(
            all_refs,
            all_gt_centers,
            prefix="all_gt_all_proposals",
        )
    )
    summary.update(
        proposal_set_metrics(
            learned_refs,
            all_gt_centers,
            prefix="all_gt_learned",
        )
    )
    summary.update(
        proposal_set_metrics(
            all_refs,
            source9_gt_centers,
            prefix="source9_all_proposals",
        )
    )
    summary.update(
        proposal_set_metrics(
            learned_refs,
            source9_gt_centers,
            prefix="source9_learned",
        )
    )

    rows = []
    gt_id_to_row = {
        int(gt_id): row
        for row, gt_id in enumerate(all_gt_ids.tolist())
    }
    for gt_id in missing_gt_ids:
        gt_row = gt_id_to_row[int(gt_id)]
        center = all_gt_centers[gt_row : gt_row + 1]

        rows.append(
            {
                "gt_id": int(gt_id),
                "nearest_learned_proposal_dref": (
                    float(torch.cdist(learned_refs, center).min())
                    if len(learned_refs)
                    else float("inf")
                ),
                "nearest_any_proposal_dref": (
                    float(torch.cdist(all_refs, center).min())
                    if len(all_refs)
                    else float("inf")
                ),
            }
        )

    frame = pd.DataFrame(rows)
    frame.to_csv(
        RUN_DIR / "O04_G4_missing_gt_proposals.csv",
        index=False,
    )
    pd.DataFrame([summary]).to_csv(
        RUN_DIR / "O04_G4_proposal_summary.csv",
        index=False,
    )
    return summary, frame


gate4_result = safe_phase(
    "O04_G4_proposal_locations",
    _gate4_proposal_locations,
)
if gate4_result is not None:
    gate4_summary, missing_gt_proposal_df = gate4_result
    display(pd.DataFrame([gate4_summary]))
    if len(missing_gt_proposal_df):
        display(missing_gt_proposal_df)
    record_result(
        "O04_G4_proposal_locations",
        **gate4_summary,
    )


## G5 — Learned existence / cardinality

In [ ]:
def binary_auc_small(scores: np.ndarray, labels: np.ndarray) -> float:
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=bool)

    positive = scores[labels]
    negative = scores[~labels]
    if len(positive) == 0 or len(negative) == 0:
        return float("nan")

    comparisons = (
        (positive[:, None] > negative[None, :]).astype(float)
        + 0.5
        * (positive[:, None] == negative[None, :]).astype(float)
    )
    return float(comparisons.mean())


gate5_sweep_df = pd.DataFrame()
gate5_summary = None

def _gate5_selection():
    sweep = [
        selection_metrics_for_source9(
            out_full,
            threshold,
        )
        for threshold in (
            0.10,
            0.20,
            0.30,
            0.40,
            0.50,
            0.60,
            0.70,
            0.80,
            0.90,
        )
    ]
    frame = pd.DataFrame(sweep)

    qrows = valid_proposal_query_rows(out_full)
    scores = (
        out_full.exist_logits[0, qrows]
        .sigmoid()
        .detach()
        .cpu()
        .numpy()
    )

    positive_rows = set(
        source9_pairing["query_rows"].tolist()
    )
    labels = np.asarray(
        [
            int(row) in positive_rows
            for row in qrows.detach().cpu().tolist()
        ],
        dtype=bool,
    )
    auc = binary_auc_small(scores, labels)

    default_row = frame[
        np.isclose(
            frame["threshold"],
            DEFAULT_EXIST_THRESHOLD,
        )
    ].iloc[0].to_dict()

    summary = {
        **default_row,
        "oracle_pair_ranking_auc_all_proposals": auc,
        "total_valid_proposal_queries": int(len(qrows)),
    }

    frame.to_csv(
        RUN_DIR / "O05_G5_selection_threshold_sweep.csv",
        index=False,
    )
    pd.DataFrame([summary]).to_csv(
        RUN_DIR / "O05_G5_selection_default.csv",
        index=False,
    )
    return frame, summary


gate5_result = safe_phase(
    "O05_G5_learned_selection",
    _gate5_selection,
)
if gate5_result is not None:
    gate5_sweep_df, gate5_summary = gate5_result
    display(gate5_sweep_df)
    print(gate5_summary)
    record_result(
        "O05_G5_learned_selection",
        **gate5_summary,
    )


## G6 — Oracle proposal-score field vs learned score field

In [ ]:
def centers_to_native_voxels(
    centers_cellscale: torch.Tensor,
    shape: tuple[int, int, int],
    spacing_um: torch.Tensor,
    dref: torch.Tensor,
) -> torch.Tensor:
    centers = centers_cellscale.to(
        device=spacing_um.device,
        dtype=torch.float32,
    )
    shape_tensor = torch.tensor(
        shape,
        device=spacing_um.device,
        dtype=torch.float32,
    )
    extent = (
        shape_tensor - 1.0
    ) * spacing_um.float()

    centers_um = centers * dref.float()
    voxels = torch.round(
        (centers_um + 0.5 * extent[None])
        / spacing_um.float()[None].clamp_min(1e-8)
    ).long()

    return torch.minimum(
        torch.maximum(
            voxels,
            torch.zeros_like(voxels),
        ),
        shape_tensor.long()[None] - 1,
    )


gate6_summary = None
proposal_score_gt_df = pd.DataFrame()

def _gate6_score_field():
    score_logits = out_full.dense_outputs[
        "proposal_score_logits"
    ]
    shape = tuple(int(v) for v in score_logits.shape[-3:])
    spacing = out_full.spacing_um[0]
    dref = out_full.dref_um[0]

    gt_centers_dev = all_gt_centers.to(device)
    gt_voxels = centers_to_native_voxels(
        gt_centers_dev,
        shape,
        spacing,
        dref,
    )

    oracle_logits = torch.full(
        shape,
        -20.0,
        device=device,
        dtype=torch.float32,
    )
    oracle_logits[
        gt_voxels[:, 0],
        gt_voxels[:, 1],
        gt_voxels[:, 2],
    ] = 20.0

    padding = None
    trainer.model.eval()
    oracle_voxels = (
        trainer.model.spatial_proposal_generator._learned_centers(
            oracle_logits,
            spacing,
            dref,
            padding,
            int(cfg.proposals.max_proposals),
        )
    )
    oracle_refs_um = (
        trainer.model.spatial_proposal_generator
        ._relative_voxel_centers_um(
            oracle_voxels,
            shape,
            spacing,
        )
    )
    oracle_refs = (
        oracle_refs_um
        / dref.float().clamp_min(1e-8)
    ).detach().cpu()

    proposals = out_full.proposals
    valid = ~proposals.padding_mask[0]
    learned = valid & ~proposals.fallback_mask[0]
    learned_refs = (
        proposals.references_cellscale[
            0, learned
        ].detach().float().cpu()
    )

    summary = {}
    summary.update(
        proposal_set_metrics(
            oracle_refs,
            all_gt_centers,
            prefix="oracle_score_nms_all_gt",
        )
    )
    summary.update(
        proposal_set_metrics(
            oracle_refs,
            source9_gt_centers,
            prefix="oracle_score_nms_source9",
        )
    )
    summary.update(
        proposal_set_metrics(
            learned_refs,
            all_gt_centers,
            prefix="learned_score_peaks_all_gt",
        )
    )
    summary.update(
        proposal_set_metrics(
            learned_refs,
            source9_gt_centers,
            prefix="learned_score_peaks_source9",
        )
    )

    probability = (
        score_logits[0, 0]
        .sigmoid()
        .detach()
        .float()
        .cpu()
        .numpy()
    )

    gt_center_scores = probability[
        gt_voxels[:, 0].cpu().numpy(),
        gt_voxels[:, 1].cpu().numpy(),
        gt_voxels[:, 2].cpu().numpy(),
    ]

    rows = []
    missing_set = set(missing_gt_ids)
    for gt_id, score in zip(
        all_gt_ids.tolist(),
        gt_center_scores.tolist(),
    ):
        rows.append(
            {
                "gt_id": int(gt_id),
                "score_at_gt_center": float(score),
                "missing_from_current_mask": int(gt_id) in missing_set,
                "source9": int(gt_id) in source9_id_set,
            }
        )

    frame = pd.DataFrame(rows)

    # Quantiles are evaluated on a bounded random background sample to avoid
    # making another large native-resolution copy.
    background_values = probability[gt_labels_native == 0]
    if background_values.size:
        rng = np.random.default_rng(SEED)
        sample_count = min(
            500_000,
            background_values.size,
        )
        if sample_count < background_values.size:
            rows_idx = rng.choice(
                background_values.size,
                size=sample_count,
                replace=False,
            )
            background_values = background_values[rows_idx]

        summary.update(
            {
                "background_score_q50": float(
                    np.quantile(background_values, 0.50)
                ),
                "background_score_q90": float(
                    np.quantile(background_values, 0.90)
                ),
                "background_score_q99": float(
                    np.quantile(background_values, 0.99)
                ),
                "gt_center_score_mean": float(
                    np.mean(gt_center_scores)
                ),
                "gt_center_score_min": float(
                    np.min(gt_center_scores)
                ),
            }
        )

    frame.to_csv(
        RUN_DIR / "O06_G6_proposal_scores_at_gt.csv",
        index=False,
    )
    pd.DataFrame([summary]).to_csv(
        RUN_DIR / "O06_G6_score_field_summary.csv",
        index=False,
    )
    return summary, frame


gate6_result = safe_phase(
    "O06_G6_proposal_score_field",
    _gate6_score_field,
)
if gate6_result is not None:
    gate6_summary, proposal_score_gt_df = gate6_result
    display(pd.DataFrame([gate6_summary]))
    display(proposal_score_gt_df)
    record_result(
        "O06_G6_proposal_score_field",
        **gate6_summary,
    )


## G7 — Missing-cell discovery and noisy-current suppression

In [ ]:
missing_discovery_df = pd.DataFrame()
noisy_suppression_df = pd.DataFrame()
gate7_summary = None

def _gate7_missing_and_noise():
    qrows = valid_proposal_query_rows(out_full)
    refs = (
        out_full.query_initial_references_cellscale[
            0, qrows
        ].detach().float().cpu()
    )
    scores = (
        out_full.exist_logits[
            0, qrows
        ].sigmoid().detach().float().cpu()
    )
    source_ids = (
        out_full.source_instance_ids[
            0, qrows
        ].detach().cpu().long()
    )

    gt_id_to_row = {
        int(gt_id): row
        for row, gt_id in enumerate(all_gt_ids.tolist())
    }

    missing_rows = []
    for gt_id in missing_gt_ids:
        center = all_gt_centers[
            gt_id_to_row[int(gt_id)]
            : gt_id_to_row[int(gt_id)] + 1
        ]

        if len(refs):
            distance = torch.cdist(refs, center)[:, 0]
            nearest_row = int(distance.argmin())
            nearby = distance <= 1.0
            selected_nearby = nearby & (
                scores >= DEFAULT_EXIST_THRESHOLD
            )
            missing_rows.append(
                {
                    "gt_id": int(gt_id),
                    "nearest_proposal_dref": float(
                        distance[nearest_row]
                    ),
                    "nearest_exist_prob": float(
                        scores[nearest_row]
                    ),
                    "proposals_within_1dref": int(
                        nearby.sum()
                    ),
                    "selected_within_1dref": int(
                        selected_nearby.sum()
                    ),
                }
            )
        else:
            missing_rows.append(
                {
                    "gt_id": int(gt_id),
                    "nearest_proposal_dref": float("inf"),
                    "nearest_exist_prob": float("nan"),
                    "proposals_within_1dref": 0,
                    "selected_within_1dref": 0,
                }
            )

    noisy_rows = []
    for source_id in noisy_source_ids:
        tied = source_ids == int(source_id)
        selected = tied & (
            scores >= DEFAULT_EXIST_THRESHOLD
        )
        noisy_rows.append(
            {
                "source_id": int(source_id),
                "proposal_queries": int(tied.sum()),
                "selected_queries": int(selected.sum()),
                "max_exist_prob": (
                    float(scores[tied].max())
                    if bool(tied.any())
                    else float("nan")
                ),
                "mean_exist_prob": (
                    float(scores[tied].mean())
                    if bool(tied.any())
                    else float("nan")
                ),
            }
        )

    missing_frame = pd.DataFrame(missing_rows)
    noisy_frame = pd.DataFrame(noisy_rows)

    summary = {
        "missing_gt_count": len(missing_gt_ids),
        "missing_gt_with_selected_proposal_within_1dref": (
            int(
                (
                    missing_frame["selected_within_1dref"] > 0
                ).sum()
            )
            if len(missing_frame)
            else 0
        ),
        "noisy_source_count": len(noisy_source_ids),
        "noisy_sources_with_selected_query": (
            int(
                (
                    noisy_frame["selected_queries"] > 0
                ).sum()
            )
            if len(noisy_frame)
            else 0
        ),
    }

    missing_frame.to_csv(
        RUN_DIR / "O07_G7_missing_cell_discovery.csv",
        index=False,
    )
    noisy_frame.to_csv(
        RUN_DIR / "O07_G7_noisy_component_suppression.csv",
        index=False,
    )
    return missing_frame, noisy_frame, summary


gate7_result = safe_phase(
    "O07_G7_missing_and_noise",
    _gate7_missing_and_noise,
)
if gate7_result is not None:
    (
        missing_discovery_df,
        noisy_suppression_df,
        gate7_summary,
    ) = gate7_result

    if len(missing_discovery_df):
        display(missing_discovery_df)
    if len(noisy_suppression_df):
        display(noisy_suppression_df)
    print(gate7_summary)
    record_result(
        "O07_G7_missing_and_noise",
        **gate7_summary,
    )


## AUX — Source-9 composition: oracle one-per-GT vs learned existence selection

This directly tests the "many colors inside one biological cell" failure mode.

Rendering is streamed **one selected query at a time** so the diagnostic does not allocate all full-resolution query masks simultaneously.

Two compositions are compared:

1. **oracle one-per-GT selection** — GT-aware one-to-one proposal identities; existence is not used to suppress a known-real cell.
2. **learned selection** — proposal hypotheses surviving the model's default existence threshold.


In [ ]:
composition_oracle_summary = None
composition_learned_summary = None

def source9_crop_slices(margin_dref: float = 1.5):
    coords = np.argwhere(current_labels_native == SOURCE_ID)
    if len(coords) == 0:
        coords = np.argwhere(
            np.isin(gt_labels_native, source9_gt_ids)
        )

    lo = coords.min(axis=0)
    hi = coords.max(axis=0) + 1
    margin = np.ceil(
        margin_dref * dref_um / spacing_native
    ).astype(int)

    lo = np.maximum(0, lo - margin)
    hi = np.minimum(
        np.asarray(gt_labels_native.shape),
        hi + margin,
    )

    return tuple(
        slice(int(a), int(b))
        for a, b in zip(lo, hi)
    )


def compose_queries_streamed(
    outputs,
    qrows: torch.Tensor,
    *,
    use_existence: bool,
) -> np.ndarray:
    qrows = qrows.to(
        device=device,
        dtype=torch.long,
    )
    crop = source9_crop_slices()

    crop_shape = tuple(
        sl.stop - sl.start
        for sl in crop
    )
    best_score = torch.full(
        crop_shape,
        -torch.inf,
        device=device,
        dtype=torch.float32,
    )
    winner = torch.zeros(
        crop_shape,
        device=device,
        dtype=torch.int32,
    )

    for destination, qrow in enumerate(qrows.tolist(), start=1):
        selected = torch.tensor(
            [qrow],
            device=device,
            dtype=torch.long,
        )

        with torch.no_grad():
            rendered = trainer.model.render_masks(
                outputs,
                [selected],
            )[0][0]

        probability = rendered[
            crop
        ].float().sigmoid()
        valid = probability >= 0.5

        if use_existence:
            exist = float(
                outputs.exist_logits[
                    0, qrow
                ].sigmoid().detach().cpu()
            )
            score = probability * exist
        else:
            score = probability

        score = score.masked_fill(~valid, -torch.inf)
        better = score > best_score

        best_score = torch.where(
            better,
            score,
            best_score,
        )
        winner = torch.where(
            better,
            torch.full_like(winner, destination),
            winner,
        )

        del rendered, probability, score, better

    winner[~torch.isfinite(best_score)] = 0
    return winner.detach().cpu().numpy()


def fragmentation_metrics(
    predicted_crop: np.ndarray,
    tag: str,
) -> tuple[pd.DataFrame, dict]:
    crop = source9_crop_slices()
    gt = gt_labels_native[crop]

    rows = []
    for gt_id in source9_gt_ids.tolist():
        gt_mask = gt == int(gt_id)
        gt_count = int(gt_mask.sum())

        overlapping = np.unique(
            predicted_crop[gt_mask]
        )
        overlapping = overlapping[overlapping > 0]

        significant = []
        best_dice = 0.0

        for pred_id in overlapping.tolist():
            pred_mask = predicted_crop == int(pred_id)
            intersection = int(
                np.count_nonzero(gt_mask & pred_mask)
            )

            if intersection >= max(
                3,
                int(0.05 * max(gt_count, 1)),
            ):
                significant.append(int(pred_id))

            dice = (
                2.0 * intersection
                / max(
                    int(gt_mask.sum())
                    + int(pred_mask.sum()),
                    1,
                )
            )
            best_dice = max(best_dice, dice)

        rows.append(
            {
                "tag": tag,
                "gt_id": int(gt_id),
                "significant_predicted_fragments": len(significant),
                "best_composed_dice": best_dice,
                "missing": int(len(significant) == 0),
            }
        )

    frame = pd.DataFrame(rows)
    summary = {
        "tag": tag,
        "mean_best_composed_dice": float(
            frame["best_composed_dice"].mean()
        ),
        "missing_gt": int(frame["missing"].sum()),
        "gt_with_multiple_fragments": int(
            (
                frame["significant_predicted_fragments"] > 1
            ).sum()
        ),
        "mean_fragments_per_gt": float(
            frame["significant_predicted_fragments"].mean()
        ),
        "predicted_instance_count_in_crop": int(
            np.count_nonzero(
                np.unique(predicted_crop) > 0
            )
        ),
    }
    return frame, summary


def _composition_test():
    oracle_qrows = source9_pairing[
        "query_rows"
    ].to(device)

    proposal_qrows = valid_proposal_query_rows(out_full)
    proposal_refs = (
        out_full.query_initial_references_cellscale[
            0, proposal_qrows
        ].detach().float().cpu()
    )
    proposal_scores = (
        out_full.exist_logits[
            0, proposal_qrows
        ].sigmoid().detach().float().cpu()
    )

    distance = torch.cdist(
        proposal_refs,
        source9_gt_centers.float(),
    )
    near = (
        distance.min(dim=1).values
        <= SOURCE9_SELECTION_NEIGHBORHOOD_DREF
    )
    learned_local = near & (
        proposal_scores >= DEFAULT_EXIST_THRESHOLD
    )
    learned_qrows = proposal_qrows[
        learned_local.to(proposal_qrows.device)
    ]

    if len(learned_qrows) > MAX_SOURCE9_RENDER_QUERIES:
        selected_scores = out_full.exist_logits[
            0, learned_qrows
        ].sigmoid()
        top = torch.topk(
            selected_scores,
            MAX_SOURCE9_RENDER_QUERIES,
        ).indices
        learned_qrows = learned_qrows[top]

    oracle_crop = compose_queries_streamed(
        out_full,
        oracle_qrows,
        use_existence=False,
    )
    learned_crop = (
        compose_queries_streamed(
            out_full,
            learned_qrows,
            use_existence=True,
        )
        if len(learned_qrows)
        else np.zeros_like(oracle_crop)
    )

    oracle_frame, oracle_summary = fragmentation_metrics(
        oracle_crop,
        "oracle_one_per_gt",
    )
    learned_frame, learned_summary = fragmentation_metrics(
        learned_crop,
        "learned_existence_selection",
    )

    crop = source9_crop_slices()
    np.savez_compressed(
        RUN_DIR / "source9_composition.npz",
        gt_source9_crop=gt_labels_native[crop].astype(np.int32),
        current_source9_crop=current_labels_native[crop].astype(np.int32),
        oracle_composed_crop=oracle_crop.astype(np.int32),
        learned_composed_crop=learned_crop.astype(np.int32),
        oracle_query_rows=oracle_qrows.detach().cpu().numpy(),
        learned_query_rows=learned_qrows.detach().cpu().numpy(),
    )

    oracle_frame.to_csv(
        RUN_DIR / "O08_AUX_oracle_composition.csv",
        index=False,
    )
    learned_frame.to_csv(
        RUN_DIR / "O08_AUX_learned_composition.csv",
        index=False,
    )

    return (
        oracle_frame,
        oracle_summary,
        learned_frame,
        learned_summary,
    )


composition_result = safe_phase(
    "O08_AUX_source9_composition",
    _composition_test,
)
if composition_result is not None:
    (
        composition_oracle_df,
        composition_oracle_summary,
        composition_learned_df,
        composition_learned_summary,
    ) = composition_result

    display(composition_oracle_df)
    display(composition_learned_df)
    print("Oracle selection :", composition_oracle_summary)
    print("Learned selection:", composition_learned_summary)

    record_result(
        "O08_AUX_oracle_composition",
        **composition_oracle_summary,
    )
    record_result(
        "O08_AUX_learned_composition",
        **composition_learned_summary,
    )

cleanup()


## G8 — Temporal contribution: trained full model vs spatial-only ablation

The model was trained with the real temporal graph/history input.

We now compare the final checkpoint with:

- full temporal evidence,
- an otherwise identical forward where temporal nodes/tracklets are removed.

This is still an ablation, not proof of causal biological reasoning, but unlike Notebook 29 the temporal pathway has now had actual training opportunity.


In [ ]:
def compact_output_summary(outputs, tag: str) -> dict:
    pairing = build_gt_aware_pairing(
        outputs,
        source9_gt_ids,
        source9_gt_centers,
    )
    qrows = pairing["query_rows"]
    grows = pairing["gt_rows"]

    summary = {"tag": tag}

    if len(qrows):
        qrows_dev = qrows.to(device)
        paired_gt = source9_gt_centers[grows]

        initial = (
            outputs.query_initial_references_cellscale[
                0, qrows_dev
            ].detach().float().cpu()
        )
        final = (
            outputs.centers_cellscale[
                0, qrows_dev
            ].detach().float().cpu()
        )

        summary[
            "paired_initial_center_mean_dref"
        ] = float(
            torch.linalg.vector_norm(
                initial - paired_gt,
                dim=-1,
            ).mean()
        )
        summary[
            "paired_final_center_mean_dref"
        ] = float(
            torch.linalg.vector_norm(
                final - paired_gt,
                dim=-1,
            ).mean()
        )

        frame, mask_summary = eval_local_masks(
            trainer.model.local_mask_decoder,
            outputs,
            gt_ids_ordered=pairing["gt_ids"],
            anchors_cellscale=(
                outputs.query_initial_references_cellscale[
                    0, qrows_dev
                ].detach()
            ),
            query_embeddings=(
                outputs.query_embeddings[
                    0, qrows_dev
                ].detach()
            ),
            tag=f"{tag}_local_masks",
        )
        summary[
            "paired_local_hard_dice_mean"
        ] = mask_summary["hard_dice_mean"]
        summary[
            "paired_local_hard_dice_min"
        ] = mask_summary["hard_dice_min"]

    proposals = outputs.proposals
    if proposals is not None:
        valid = ~proposals.padding_mask[0]
        refs = (
            proposals.references_cellscale[
                0, valid
            ].detach().float().cpu()
        )
        summary.update(
            proposal_set_metrics(
                refs,
                source9_gt_centers,
                prefix="source9",
            )
        )

    selection = selection_metrics_for_source9(
        outputs,
        DEFAULT_EXIST_THRESHOLD,
    )
    summary.update(
        {
            f"selection_{key}": value
            for key, value in selection.items()
            if key != "threshold"
        }
    )
    return summary


gate8_df = pd.DataFrame()
gate8_summary_full = None
gate8_summary_spatial = None

def _gate8_temporal_ablation():
    global out_full

    full_summary = compact_output_summary(
        out_full,
        "full_temporal",
    )

    # Free the large full-resolution snapshot before constructing the second.
    del out_full
    out_full = None
    cleanup()

    spatial_batch_cpu = make_spatial_only_batch(batch_cpu)
    gpu_spatial = move_batch_to_device(
        spatial_batch_cpu,
        device,
    )

    with torch.no_grad(), torch.autocast(
        device_type="cuda",
        dtype=AMP_DTYPE,
    ):
        spatial_outputs = model_forward_from_batch(
            trainer.model,
            gpu_spatial,
            return_debug=True,
        )

    spatial_summary = compact_output_summary(
        spatial_outputs,
        "spatial_only",
    )

    del spatial_outputs, gpu_spatial
    cleanup()

    frame = pd.DataFrame(
        [full_summary, spatial_summary]
    )
    frame.to_csv(
        RUN_DIR / "O09_G8_temporal_ablation.csv",
        index=False,
    )
    return full_summary, spatial_summary, frame


gate8_result = safe_phase(
    "O09_G8_temporal_ablation",
    _gate8_temporal_ablation,
)
if gate8_result is not None:
    (
        gate8_summary_full,
        gate8_summary_spatial,
        gate8_df,
    ) = gate8_result
    display(gate8_df)
    record_result(
        "O09_G8_temporal_ablation",
        full=gate8_summary_full,
        spatial=gate8_summary_spatial,
    )


## Automatic trained-system diagnosis

In [ ]:
diagnosis_rows = []


def add_diag(stage, verdict, evidence, next_action):
    diagnosis_rows.append(
        {
            "stage": stage,
            "verdict": verdict,
            "evidence": evidence,
            "next_action": next_action,
        }
    )


# G1 — architecture capacity
if gate1_summary is None:
    add_diag(
        "G1 local-mask capacity",
        "NOT INTERPRETABLE",
        "Oracle capacity probe failed or did not run.",
        "Inspect the G1 error before redesigning upstream proposal logic.",
    )
else:
    mean_dice = gate1_summary["hard_dice_mean"]
    min_dice = gate1_summary["hard_dice_min"]

    if mean_dice >= 0.85 and min_dice >= 0.60:
        add_diag(
            "G1 local-mask capacity",
            "PASS",
            f"GT anchors + common query capacity fit reached mean hard Dice "
            f"{mean_dice:.3f}, min {min_dice:.3f}.",
            "The local spatial evidence/decoder architecture can represent the cells; continue downstream diagnosis.",
        )
    else:
        add_diag(
            "G1 local-mask capacity",
            "SUPPORTED PROBLEM",
            f"Even with GT anchors, the temporary common-query capacity fit reached "
            f"mean hard Dice {mean_dice:.3f}, min {min_dice:.3f}.",
            "Re-examine the local spatial evidence/decoder interface before proposal or temporal redesign.",
        )


# G2 — trained query + perfect location
if gate2_summary is None:
    add_diag(
        "G2 trained query at GT anchor",
        "NOT INTERPRETABLE",
        "Gate 2 failed.",
        "Fix the trained-query oracle diagnostic.",
    )
else:
    mean_dice = gate2_summary["hard_dice_mean"]
    if mean_dice >= 0.75:
        add_diag(
            "G2 trained query at GT anchor",
            "PASS",
            f"Trained actual queries at perfect GT anchors reached mean hard Dice {mean_dice:.3f}.",
            "Query-conditioned masking is usable; inspect predicted anchors.",
        )
    else:
        add_diag(
            "G2 trained query at GT anchor",
            "SUPPORTED PROBLEM",
            f"Trained actual queries at perfect GT anchors reached only mean hard Dice {mean_dice:.3f}.",
            "The trained query/local-mask interface is weak even when localization is perfect.",
        )


# G3 — proposal anchor and center refinement
if gate3_summary is None:
    add_diag(
        "G3 predicted proposal anchors",
        "NOT INTERPRETABLE",
        "Gate 3 failed.",
        "Fix Gate 3 before changing proposal localization.",
    )
else:
    g3 = gate3_summary["hard_dice_mean"]
    coverage = gate3_summary["support_coverage_min"]
    g2 = (
        gate2_summary["hard_dice_mean"]
        if gate2_summary is not None
        else float("nan")
    )
    anchor_drop = (
        g2 - g3
        if math.isfinite(g2)
        else float("nan")
    )

    if (
        g3 >= 0.65
        and coverage >= 0.98
        and (
            not math.isfinite(anchor_drop)
            or anchor_drop <= 0.10
        )
    ):
        add_diag(
            "G3 predicted proposal anchors",
            "PASS",
            f"Predicted anchors retained hard Dice {g3:.3f}, "
            f"support coverage min {coverage:.3f}, GT-anchor drop {anchor_drop:.3f}.",
            "Proposal localization is sufficient for masking; inspect cardinality and composition.",
        )
    else:
        add_diag(
            "G3 predicted proposal anchors",
            "SUPPORTED PROBLEM",
            f"Predicted anchors produced hard Dice {g3:.3f}, "
            f"support coverage min {coverage:.3f}, GT-anchor drop {anchor_drop:.3f}.",
            "Localization/support materially degrades the trained local masks.",
        )

    initial_error = gate3_summary[
        "initial_center_error_mean_dref"
    ]
    final_error = gate3_summary[
        "final_center_error_mean_dref"
    ]

    add_diag(
        "G3b center refinement identity",
        (
            "PASS"
            if final_error <= initial_error + 0.05
            else "SUPPORTED PROBLEM"
        ),
        f"Paired center error {initial_error:.3f} -> {final_error:.3f} dref.",
        (
            "Anchor-relative refinement preserves same-cell identity."
            if final_error <= initial_error + 0.05
            else "Center refinement is still moving paired hypotheses away from their own GT identities."
        ),
    )


# G4
if gate4_summary is None:
    add_diag(
        "G4 proposal coverage",
        "NOT INTERPRETABLE",
        "Proposal gate failed.",
        "Fix proposal diagnostics.",
    )
else:
    s9 = gate4_summary.get(
        "source9_all_proposals_recall_0p5",
        float("nan"),
    )
    all_recall = gate4_summary.get(
        "all_gt_all_proposals_recall_0p5",
        float("nan"),
    )

    add_diag(
        "G4 proposal coverage",
        (
            "PASS"
            if s9 >= 0.999 and all_recall >= 0.90
            else "SUPPORTED PROBLEM"
        ),
        f"Recall@0.5 dref: source9={s9:.3f}, all GT={all_recall:.3f}.",
        (
            "High-recall proposal generation is adequate."
            if s9 >= 0.999 and all_recall >= 0.90
            else "The proposal pool itself is missing usable cell hypotheses."
        ),
    )


# G5
if gate5_summary is None:
    add_diag(
        "G5 learned cardinality",
        "NOT INTERPRETABLE",
        "Selection gate failed.",
        "Fix learned-selection diagnostics.",
    )
else:
    unique = int(gate5_summary["unique_gt_covered"])
    missing = int(gate5_summary["missing_gt"])
    duplicates = int(gate5_summary["duplicate_selected"])

    add_diag(
        "G5 learned cardinality",
        (
            "PASS"
            if unique == len(source9_gt_ids)
            and missing == 0
            and duplicates <= 1
            else "SUPPORTED PROBLEM"
        ),
        f"Default threshold covers {unique}/{len(source9_gt_ids)}, "
        f"missing={missing}, duplicate survivors={duplicates}.",
        (
            "Existence/cardinality is provisionally adequate."
            if unique == len(source9_gt_ids)
            and missing == 0
            and duplicates <= 1
            else "Existence/cardinality is a direct candidate for instance fragmentation or missing cells."
        ),
    )


# G6
if gate6_summary is None:
    add_diag(
        "G6 proposal score field / NMS",
        "NOT INTERPRETABLE",
        "Score-field gate failed.",
        "Fix the score-field diagnostic.",
    )
else:
    oracle_recall = gate6_summary.get(
        "oracle_score_nms_all_gt_recall_0p5",
        float("nan"),
    )
    learned_recall = gate6_summary.get(
        "learned_score_peaks_all_gt_recall_0p5",
        float("nan"),
    )

    if oracle_recall < 0.99:
        verdict = "SUPPORTED PROBLEM"
        action = (
            "NMS/extraction geometry itself suppresses legitimate oracle peaks."
        )
    elif learned_recall + 0.05 < oracle_recall:
        verdict = "SUPPORTED PROBLEM"
        action = (
            "NMS can work, but the learned proposal score field is the weak link."
        )
    else:
        verdict = "PASS"
        action = "Proposal score/extraction is provisionally adequate."

    add_diag(
        "G6 proposal score field / NMS",
        verdict,
        f"Oracle NMS recall={oracle_recall:.3f}, learned-peak recall={learned_recall:.3f}.",
        action,
    )


# G7
if gate7_summary is not None:
    if gate7_summary["missing_gt_count"] > 0:
        found = gate7_summary[
            "missing_gt_with_selected_proposal_within_1dref"
        ]
        total = gate7_summary["missing_gt_count"]
        add_diag(
            "G7a missing-cell discovery",
            "PASS" if found == total else "SUPPORTED PROBLEM",
            f"Selected nearby proposal for {found}/{total} GT cells absent from the current mask.",
            (
                "Off-mask discovery worked on this scene."
                if found == total
                else "The model still misses some cells with no current-mask support."
            ),
        )
    else:
        add_diag(
            "G7a missing-cell discovery",
            "NOT TESTED",
            "This frame contains no completely missing GT cells.",
            "Use a separate scene containing true cells with zero current-mask overlap.",
        )

    if gate7_summary["noisy_source_count"] > 0:
        survivors = gate7_summary[
            "noisy_sources_with_selected_query"
        ]
        total = gate7_summary["noisy_source_count"]
        add_diag(
            "G7b noisy-current suppression",
            "PASS" if survivors == 0 else "SUPPORTED PROBLEM",
            f"{survivors}/{total} GT-empty current components retain selected proposal queries.",
            (
                "Noisy source suppression is working."
                if survivors == 0
                else "Existence reasoning still trusts some purely noisy current components."
            ),
        )


# Composition
if (
    composition_oracle_summary is not None
    and composition_learned_summary is not None
):
    oracle_frag = composition_oracle_summary[
        "gt_with_multiple_fragments"
    ]
    learned_frag = composition_learned_summary[
        "gt_with_multiple_fragments"
    ]

    add_diag(
        "AUX final composition",
        (
            "SUPPORTED PROBLEM"
            if learned_frag > oracle_frag
            else "PASS"
        ),
        f"GT cells with multiple fragments: oracle selection={oracle_frag}, "
        f"learned selection={learned_frag}.",
        (
            "Learned survivor count/competition adds fragmentation beyond one-per-cell rendering."
            if learned_frag > oracle_frag
            else "Learned selection does not add substantial fragmentation beyond mask quality."
        ),
    )


# Temporal
if (
    gate8_summary_full is not None
    and gate8_summary_spatial is not None
):
    full_dice = float(
        gate8_summary_full.get(
            "paired_local_hard_dice_mean",
            float("nan"),
        )
    )
    spatial_dice = float(
        gate8_summary_spatial.get(
            "paired_local_hard_dice_mean",
            float("nan"),
        )
    )
    full_center = float(
        gate8_summary_full.get(
            "paired_final_center_mean_dref",
            float("nan"),
        )
    )
    spatial_center = float(
        gate8_summary_spatial.get(
            "paired_final_center_mean_dref",
            float("nan"),
        )
    )

    add_diag(
        "G8 temporal contribution",
        "DIAGNOSTIC",
        f"Local hard Dice full={full_dice:.3f}, spatial-only={spatial_dice:.3f}; "
        f"center error full={full_center:.3f}, spatial-only={spatial_center:.3f} dref.",
        "Use this trained ablation together with temporal attention/event diagnostics before deciding whether temporal fusion needs redesign.",
    )


diagnosis_df = pd.DataFrame(diagnosis_rows)
display(diagnosis_df)

diagnosis_df.to_csv(
    RUN_DIR / "diagnosis.csv",
    index=False,
)
(RUN_DIR / "diagnosis.json").write_text(
    json.dumps(
        diagnosis_rows,
        indent=2,
    ),
    encoding="utf-8",
)

priority = diagnosis_df[
    diagnosis_df["verdict"].isin(
        ["SUPPORTED PROBLEM"]
    )
]

if len(priority):
    first_problem = priority.iloc[0].to_dict()
    print()
    print("=" * 100)
    print("FIRST TRAINED-SYSTEM SUPPORTED PROBLEM")
    print("=" * 100)
    print(first_problem["stage"])
    print(first_problem["evidence"])
    print("Next:", first_problem["next_action"])
else:
    print()
    print(
        "No strong failure was isolated by the completed trained-system gates. "
        "Inspect the detailed tables and temporal diagnostics before changing architecture."
    )


## Final run-health report

In [ ]:
elapsed_minutes = (
    time.monotonic() - NOTEBOOK_STARTED_MONO
) / 60.0

health = {
    "git_head": HEAD,
    "elapsed_minutes": elapsed_minutes,
    "finished": datetime.now(SRI_LANKA_TZ),
    "target_end": target_end,
    "training_rows": len(training_rows),
    "stage_summaries": stage_summaries,
    "errors_logged": (
        sum(
            1
            for _ in ERRORS_JSONL.open(
                "r",
                encoding="utf-8",
            )
        )
        if ERRORS_JSONL.exists()
        else 0
    ),
    "warm_start": warm_start_info,
    "final_local_train_cap": int(
        trainer.model.local_mask_decoder.cfg.train_max_queries_per_batch
    ),
    "run_dir": str(RUN_DIR),
}

(RUN_DIR / "run_health.json").write_text(
    json.dumps(
        {
            key: _jsonable(value)
            for key, value in health.items()
        },
        indent=2,
    ),
    encoding="utf-8",
)

(RUN_DIR / "RUN_COMPLETE.json").write_text(
    json.dumps(
        {
            "completed_at": datetime.now(
                SRI_LANKA_TZ
            ).isoformat(),
            "elapsed_minutes": elapsed_minutes,
        },
        indent=2,
    ),
    encoding="utf-8",
)

write_heartbeat(
    status="complete",
    elapsed_minutes=elapsed_minutes,
    run_dir=RUN_DIR,
)

print("=" * 100)
print("NOTEBOOK 30 COMPLETE")
print("=" * 100)
print(
    json.dumps(
        {
            key: _jsonable(value)
            for key, value in health.items()
            if key != "stage_summaries"
        },
        indent=2,
    )
)

print()
print("Key outputs:")
for name in (
    "checkpoint_start.pt",
    "checkpoint_after_query.pt",
    "checkpoint_after_local_mask.pt",
    "checkpoint_joint_last.pt",
    "checkpoint_evening_final.pt",
    "checkpoint_recovery.pt",
    "training.csv",
    "diagnosis.csv",
    "O01_G1_capacity_per_cell.csv",
    "O02_G2_actual_query_gt_anchor.csv",
    "O03_G3_predicted_anchor_masks.csv",
    "O03_G3_center_identity.csv",
    "O04_G4_proposal_summary.csv",
    "O05_G5_selection_threshold_sweep.csv",
    "O06_G6_score_field_summary.csv",
    "O07_G7_missing_cell_discovery.csv",
    "O07_G7_noisy_component_suppression.csv",
    "source9_composition.npz",
    "O09_G8_temporal_ablation.csv",
    "errors.jsonl",
    "run.log",
):
    path = RUN_DIR / name
    if path.exists():
        print(" -", path)


## Optional Napari inspection

The unattended run never opens a GUI.

After completion, `source9_composition.npz` contains:

- source-9 GT labels,
- source-9 current segmentation,
- oracle one-proposal-per-GT composition,
- learned-existence composition.

Set `OPEN_NAPARI_AT_END = True` and run the next cell manually if desired.


In [ ]:
if OPEN_NAPARI_AT_END:
    try:
        import napari

        payload = np.load(
            RUN_DIR / "source9_composition.npz"
        )

        viewer = napari.Viewer(
            title="STIR-Net Notebook 30 — trained source9 oracle ladder"
        )
        viewer.add_labels(
            payload["gt_source9_crop"],
            name="GT",
            scale=tuple(spacing_native),
        )
        viewer.add_labels(
            payload["current_source9_crop"],
            name="Current segmentation",
            scale=tuple(spacing_native),
        )
        viewer.add_labels(
            payload["oracle_composed_crop"],
            name="Oracle one-per-GT composition",
            scale=tuple(spacing_native),
        )
        viewer.add_labels(
            payload["learned_composed_crop"],
            name="Learned existence composition",
            scale=tuple(spacing_native),
        )
        viewer.dims.ndisplay = 3
    except BaseException as exc:
        record_error("optional_napari", exc)
